<a href="https://colab.research.google.com/github/saketh-varma06/DL_ASSIGNMENTS/blob/main/DL_ASS6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

classes = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print("\nDataset loaded successfully!")
print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

train_size = 45000
validation_size = 5000

train_data, validation_data = random_split(
    train_dataset,
    [train_size, validation_size],
    generator=torch.Generator().manual_seed(42)
)

print("Training samples:", len(train_data))
print("Validation samples:", len(validation_data))

batch_size = 128

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

validation_loader = DataLoader(
    validation_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

def show_images():

    images, labels = next(iter(train_loader))

    images = images * torch.tensor(
        [0.2470, 0.2435, 0.2616]
    ).view(1, 3, 1, 1)

    images = images + torch.tensor(
        [0.4914, 0.4822, 0.4465]
    ).view(1, 3, 1, 1)

    images = torch.clamp(images, 0, 1)

    plt.figure(figsize=(12, 8))

    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].permute(1, 2, 0))
        plt.title(classes[labels[i]])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_images()

class CIFAR10_CNN(nn.Module):

    def __init__(self):

        super(CIFAR10_CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

def train_model(model, optimizer, epochs=10):

    criterion = nn.CrossEntropyLoss()

    train_losses = []
    validation_losses = []
    train_accuracies = []
    validation_accuracies = []

    start_time = time.time()

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / total
        epoch_train_accuracy = 100 * correct / total

        model.eval()

        validation_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in validation_loader:

                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)

                loss = criterion(outputs, labels)

                validation_loss += loss.item() * images.size(0)

                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        epoch_validation_loss = validation_loss / total
        epoch_validation_accuracy = 100 * correct / total

        train_losses.append(epoch_train_loss)
        validation_losses.append(epoch_validation_loss)
        train_accuracies.append(epoch_train_accuracy)
        validation_accuracies.append(epoch_validation_accuracy)

        print(
            f"Epoch [{epoch + 1}/{epochs}] | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Train Acc: {epoch_train_accuracy:.2f}% | "
            f"Val Loss: {epoch_validation_loss:.4f} | "
            f"Val Acc: {epoch_validation_accuracy:.2f}%"
        )

    total_time = time.time() - start_time

    print(f"\nTraining time: {total_time / 60:.2f} minutes")

    history = {
        "train_loss": train_losses,
        "val_loss": validation_losses,
        "train_acc": train_accuracies,
        "val_acc": validation_accuracies
    }

    return history

def evaluate_model(model):

    model.eval()

    correct = 0
    total = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(
                predicted.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    accuracy = 100 * correct / total

    return accuracy, all_labels, all_predictions

print("\n")
print("=" * 60)
print("TRAINING MODEL USING SGD")
print("=" * 60)

sgd_model = CIFAR10_CNN().to(device)

sgd_optimizer = optim.SGD(
    sgd_model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=5e-4
)

sgd_history = train_model(
    sgd_model,
    sgd_optimizer,
    epochs=10
)

sgd_accuracy, sgd_labels, sgd_predictions = evaluate_model(
    sgd_model
)

print("\nSGD Test Accuracy:", f"{sgd_accuracy:.2f}%")

print("\n")
print("=" * 60)
print("TRAINING MODEL USING ADAM")
print("=" * 60)

adam_model = CIFAR10_CNN().to(device)

adam_optimizer = optim.Adam(
    adam_model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

adam_history = train_model(
    adam_model,
    adam_optimizer,
    epochs=10
)

adam_accuracy, adam_labels, adam_predictions = evaluate_model(
    adam_model
)

print("\nAdam Test Accuracy:", f"{adam_accuracy:.2f}%")

epochs_range = range(1, 11)

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    sgd_history["train_loss"],
    marker='o',
    label="SGD"
)

plt.plot(
    epochs_range,
    adam_history["train_loss"],
    marker='o',
    label="Adam"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss: SGD vs Adam")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    sgd_history["val_loss"],
    marker='o',
    label="SGD"
)

plt.plot(
    epochs_range,
    adam_history["val_loss"],
    marker='o',
    label="Adam"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Validation Loss: SGD vs Adam")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    sgd_history["train_acc"],
    marker='o',
    label="SGD"
)

plt.plot(
    epochs_range,
    adam_history["train_acc"],
    marker='o',
    label="Adam"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training Accuracy: SGD vs Adam")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_range,
    sgd_history["val_acc"],
    marker='o',
    label="SGD"
)

plt.plot(
    epochs_range,
    adam_history["val_acc"],
    marker='o',
    label="Adam"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Validation Accuracy: SGD vs Adam")
plt.legend()
plt.grid(True)
plt.show()

print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT - SGD")
print("=" * 60)

print(
    classification_report(
        sgd_labels,
        sgd_predictions,
        target_names=classes
    )
)

print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT - ADAM")
print("=" * 60)

print(
    classification_report(
        adam_labels,
        adam_predictions,
        target_names=classes
    )
)

cm_sgd = confusion_matrix(
    sgd_labels,
    sgd_predictions
)

plt.figure(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_sgd,
    display_labels=classes
)

disp.plot(
    xticks_rotation=45,
    cmap="Blues"
)

plt.title("Confusion Matrix - SGD")
plt.show()

cm_adam = confusion_matrix(
    adam_labels,
    adam_predictions
)

plt.figure(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_adam,
    display_labels=classes
)

disp.plot(
    xticks_rotation=45,
    cmap="Greens"
)

plt.title("Confusion Matrix - Adam")
plt.show()

print("\n")
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

print(f"SGD Test Accuracy  : {sgd_accuracy:.2f}%")
print(f"Adam Test Accuracy : {adam_accuracy:.2f}%")

print(
    f"\nSGD Final Training Loss  : "
    f"{sgd_history['train_loss'][-1]:.4f}"
)

print(
    f"Adam Final Training Loss : "
    f"{adam_history['train_loss'][-1]:.4f}"
)

print(
    f"\nSGD Final Validation Accuracy  : "
    f"{sgd_history['val_acc'][-1]:.2f}%"
)

print(
    f"Adam Final Validation Accuracy : "
    f"{adam_history['val_acc'][-1]:.2f}%"
)

print("\n")
print("=" * 60)
print("CONCLUSION")
print("=" * 60)

if adam_accuracy > sgd_accuracy:

    print("Adam achieved higher test accuracy than SGD.")
    print(
        "Adam generally converged faster because it uses "
        "adaptive learning rates for individual parameters."
    )

elif sgd_accuracy > adam_accuracy:

    print("SGD achieved higher test accuracy than Adam.")
    print(
        "Although Adam may converge faster initially, "
        "SGD can sometimes provide better generalization."
    )

else:

    print("Both optimizers achieved similar test accuracy.")

print(
    "\nThe convergence plots can be used to compare the "
    "speed and stability of both optimizers."
)

Device: cpu


  3%|▎         | 4.55M/170M [00:34<21:42, 127kB/s]